# rlmflow coding-agent walkthrough

Build a recursive coding agent, run a task, save the run, and embed the generated artifact inline. This mirrors `examples/coding/agent.py` but as a notebook so you can poke at every step.

Sibling notebooks read offline against a saved run directory under `examples/_runs/word-search/baseline/`:

- [`node_basics.ipynb`](./node_basics.ipynb) — querying a run: `graph.walk`, `graph.find`, `graph.where`, `Graph.load`, …
- [`viz_walkthrough.ipynb`](./viz_walkthrough.ipynb) — visualizing a run: `tree`, `gantt`, `code_log`, mermaid / dot / d2, `report_md`, the inline Gradio viewer, etc.

Running this notebook end-to-end requires `OPENAI_API_KEY` and live LLM calls. Skip ahead to the other notebooks if you just want to consume a saved run.

## 1. Build the agent

The whole run is one `Flow` driving one `Graph`:

- `Flow` — wires an LLM client (plus optional cheaper alternates registered as `fast`) to a stateful REPL. constructing a `Graph(query=query)` seeds the run; `async for event in flow.run_streaming(graph=graph)` then drives the whole tree, mutating that same `Graph` in place and yielding one `Event` per tick. (`flow.run(query=query)` / `await flow.arun(query=query)` wrap this when you only want the final result.)
- `Runtime` — *where* code runs and *which* tools it has. A `LocalRuntime` runs in-process; `runtime.register_tools(FILE_TOOLS)` exposes the filesystem tools (`read_file`, `write_file`, `edit_file`, `ls`, `grep`, …) to every agent and child — no `Flow` subclass.
- `working_directory` — set it on the runtime and agent code (and the file tools) run inside it; no manual `chdir`. Swap `LocalRuntime` for `DockerRuntime(image, working_directory=...)` to sandbox each step with the same interface.

In [1]:
from pathlib import Path
import shutil
from rflow.clients import OpenAIClient
from rflow import FILE_TOOLS, Flow, Graph, LocalRuntime


def build_agent(
    workdir: Path | str,
    max_depth: int = 2,
    max_iters: int = 30,
    workers: int = 8,
) -> Flow:
    """Construct a coding agent identical to examples/coding/agent.py."""
    # The runtime owns where code runs and which tools it has. LocalRuntime runs
    # in-process with the cwd switched into workdir; register_tools exposes the
    # file tools to every agent and child. Swap in DockerRuntime to sandbox.
    runtime = LocalRuntime(working_directory=workdir)
    runtime.register_tools(FILE_TOOLS)
    return Flow(
        OpenAIClient("gpt-5"),
        llm_clients={"fast": OpenAIClient("gpt-5-mini")},
        runtime=runtime,
        max_depth=max_depth,
        max_iters=max_iters,
        workers=workers,
    )

## 2. Run a task

Constructing a `Graph(query=query)` seeds the live graph (just the root agent with its `UserQuery` node). `agent.run_streaming(graph=graph)` then drives the whole tree, mutating that same `Graph` in place and yielding one `Event` per tick — one LLM call, one REPL execution, or a wait-resolution. Use `graph.current()` to inspect the latest node, `graph.finished` to check for completion, and `graph.result()` for the terminal answer. `replay(graph)` reconstructs per-step snapshots afterward for a timeline.

The agent loop and its side effects are decoupled via **consumers**. Each event is fanned out through a `ConsumerGroup` to small `StreamConsumer` objects — here a notebook renderer that redraws `render_tree(graph)` in place and a `GraphCheckpointer` that persists the run to disk on every tick (so a crash still leaves the latest graph saved). The graph stays the durable source of truth.

In [2]:
TASK = """Create a runnable browser-based boids simulation in plain HTML, CSS, and JavaScript.
Requirements:
- The main runnable interface is `index.html`.
- Write separate files:
    - `index.html`
    - `style.css`
    - `boids.js`
- Do not use build tools or external libraries.
- Use a dark color background.
- Do not use ES modules; wire scripts with `<script src="..."></script>` tags.
- Render 100s of colorful boids on a 2D canvas. Do not add configurations, just the canvas.
- Verify that all files exist, script tags are ordered correctly, and the JavaScript has no obvious syntax/runtime wiring errors before returning.
"""
WORKDIR = Path("../_runs/notebooks/boids-sim").resolve()
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir(parents=True, exist_ok=True)

agent = build_agent(WORKDIR, max_depth=2)

In [3]:
graph = Graph(query=TASK)
# Built from the live prompt builder so the preview tracks prompt changes.
prompt = agent.build_system_prompt(graph)
print(prompt)

You are a Recursive Coding Agent: a language model with a prompt and important inputs stored in a persistent Python REPL. You will be queried turn by turn until you have solved the task. Use the REPL to inspect inputs, run code, query sub-LLMs, and spawn recursive sub-agents; when finished, call `done(...)`.

Available in the REPL:

1. `INPUTS`: a dict of string inputs. `INPUTS["query"]` is always the current prompt. Every other key is caller-defined; inspect `list(INPUTS)` instead of assuming names. Use `len(...)`, `.splitlines()`, and short slices for orientation. JSON inputs can be parsed with `json.loads(INPUTS["key"])`. Keys never shadow REPL variables or tools.
2. `llm_query_batched(prompts, *, model="default", output_schema=None, temperature=None, top_p=None, max_tokens=None, stop=None) -> list`: concurrent one-shot LLM calls. Use for extraction, summarization, classification, or Q&A over independent chunks. Without `output_schema`, returns `list[str]`; with a JSON Schema dict, 

In [4]:
from rflow import ConsumerGroup, GraphCheckpointer, StreamConsumer, render_tree, replay
from IPython.display import clear_output


# Consumers react to each streamed event; `run_streaming` mutates `graph` in place.
# In a notebook we redraw with clear_output instead of the ANSI-clearing
# LiveTreeRenderer used by the CLI.
class NotebookTree(StreamConsumer):
    def handle(self, event, graph):
        if graph is None:
            return
        clear_output(wait=True)
        print(render_tree(graph))


consumers = ConsumerGroup(
    [
        NotebookTree(),
        GraphCheckpointer(WORKDIR / "graph"),  # checkpoint the graph every tick
    ]
)

try:
    async for event in agent.run_streaming(graph=graph):
        consumers.handle(event, graph)
finally:
    consumers.close()  # final flush of the checkpoint

graphs = replay(graph)  # per-step snapshots for the summary + viewer below

Output()

In [ ]:
# Actual rlm run

In [5]:
current = graph.current()
print(f"{len(graphs)} snapshots  \u00b7  final: {graph.agent_id} [{current.type}]")
print(f"query : {graph.query[:120]!r}...")
print(f"result: {graph.result()[:200]}")

1 snapshots  ·  final: root [done_output]
query : 'Create a runnable browser-based boids simulation in plain HTML, CSS, and JavaScript.\nRequirements:\n- The main runnable i'...
result: Created a runnable browser-based boids simulation.
- Open index.html in a modern browser to run.
- Dark background, 2D canvas, hundreds of colorful boids rendered.
- No external libraries or build too


In [6]:
print(render_tree(graph))

● root (gpt-5) — Create a runnable browser-based boids simulation in plain H…
  - [ 0] query
  - [ 1] llm
  - [ 2] llm code=print(list(INPUTS)) print("query chars"…
  - [ 3] exec
  - [ 4] exec REPL output: ['query'] query chars 621 Create a runnable br…
  - [ 5] llm
  - [ 6] llm code=# Inspect the rest of the query (small,…
  - [ 7] exec
  - [ 8] done -> Created a runnable browser-based boids simulation. - Open i…


In [7]:
# The GraphCheckpointer already saved during the run; this is the same call it
# makes — a run directory (manifest + nested per-agent logs) alongside the files
# the agent wrote into WORKDIR. Reload any run dir with `Graph.load(path)`.
run_dir = graph.save(WORKDIR / "graph", metadata={"task": TASK})
print(f"run -> {run_dir}  ({len(graph.agents)} agents)")

run -> /Users/shyam/Code/rlmkit/examples/_runs/notebooks/boids-sim/graph  (1 agents)


## 3. Preview the generated artifact

Serve the generated `index.html` over local HTTP and embed that URL in the notebook. This exercises the same browser module-loading rules as opening the artifact normally, so import/export mistakes are not hidden by `srcdoc` inlining.

In [ ]:
# from functools import partial
# from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
# from IPython.display import IFrame
# import socket
# import threading

# candidates = sorted(WORKDIR.glob("**/index.html"))
# if not candidates:
#     raise FileNotFoundError(f"no index.html under {WORKDIR}")

# html_path = candidates[-1].resolve()
# site_root = html_path.parent

# if "preview_server" in globals():
#     preview_server.shutdown()

# with socket.socket() as s:
#     s.bind(("127.0.0.1", 0))
#     port = s.getsockname()[1]

# class QuietHandler(SimpleHTTPRequestHandler):
#     def log_message(self, *args):
#         pass

# handler = partial(QuietHandler, directory=str(site_root))
# preview_server = ThreadingHTTPServer(("127.0.0.1", port), handler)
# threading.Thread(target=preview_server.serve_forever, daemon=True).start()

# url = f"http://127.0.0.1:{port}/{html_path.name}"
# print(f"serving {site_root} -> {url}")
# IFrame(url, width="100%", height=600)

serving /Users/shyam/Code/rlmkit/examples/_runs/notebooks/boids-sim -> http://127.0.0.1:53607/index.html


## 4. Open the interactive viewer

`open_viewer(source)` boots a Gradio stepper (step slider + clickable graph + per-agent transcript) inline.

In [ ]:
from rflow.view import open_viewer

open_viewer(WORKDIR / "graph", inline=True, quiet=True)

## 5. Render frames of each step

In [ ]:
# save_steps(...) below is the supported path: it writes one PNG per step into
# WORKDIR/frames (handy for GIFs / blog strips). save_image(...) writes a single frame.

In [ ]:
from rflow.view import save_steps

save_steps(
    WORKDIR / "graph",
    WORKDIR / "frames",
    width=1600,
    height=1200,
    scale=2,
    marker_mult=3.5,
    text_mult=2.2,
)

## Next

- [`node_basics.ipynb`](./node_basics.ipynb) — walk the saved trace with the `Graph` query API.
- [`viz_walkthrough.ipynb`](./viz_walkthrough.ipynb) — inline Plotly, Mermaid, DOT, Gantt, report exports.